# Qwen3.8-Flash-Next Coder IQ1_M en Kaggle 2x T4 — **optimizado**

Versión optimizada de [qwen3.8-flash-strata-kaggle](https://github.com/jigs10/qwen3.8-flash-strata-kaggle).
Mismo modelo y mismo hardware (Kaggle gratis, **GPU T4 x2**), pero con **MTP encendido** y el resto de
ajustes puestos a favor de la velocidad.

| | Original | Este notebook |
|---|---|---|
| MTP | `--mtp off` | **`--mtp on`** + `--spec 4 --spec-min-p 0.5` + `--mtp-q4 all` |
| Expertos fríos | se leen del SSD en cada token | `--resident-budget-gib` los deja en RAM |
| Arranque | `Popen` + `sleep` + mirar el log | `strata launch --no-tui --yes`, idempotente |
| Disco | matar a mano antes de 85 GB | guarda automática |
| RAM | sin control | guarda automática contra el OOM |
| Validación | leer el plan a ojo | valida el plan **y** los flags reales del motor |
| Benchmark | una pasada | calentamiento + 3 pasadas, mediana |
| Acceso remoto | solo `127.0.0.1` | túnel de **Cloudflare** con API key |

> **Por qué MTP es la palanca grande:** el pack pesa ~58 GB y Kaggle da 32 GB de VRAM + ~31 GB de RAM.
> El motor es híbrido y lee pesos de RAM en cada paso, así que el decode está limitado por ancho de banda,
> no por cómputo. MTP propone hasta 4 tokens y el modelo completo los verifica en la misma pasada:
> más tokens por cada lectura de pesos. MTP **no cambia las respuestas** (solo propone; el modelo verifica).

Ejecuta las celdas en orden. La sección 5 valida el plan (debe decir **Coder IQ1_M**) y la 6 arranca. El túnel de
Cloudflare se prepara en la sección 4 y se muestra en la 8.


## 1. Configuración

Todo lo que se puede tocar está aquí. Las variables de entorno `SK_*` permiten cambiarlo sin editar el notebook.


In [ ]:
# ============================================================
#  CONFIG: todos los ajustes en un solo sitio
# ============================================================
import os

def _env_int(name, default):
    try:
        return int(os.environ.get(name, default))
    except ValueError:
        return default

def _env_float(name, default):
    try:
        return float(os.environ.get(name, default))
    except ValueError:
        return default

CONFIG = {
    # --- modelo: IQ1_M Coder es el unico pack que cabe en el disco de Kaggle ---
    "family": "coder",
    "model": "iq1_m",

    # --- velocidad: MTP + drafting especulativo ---
    "context": os.environ.get("SK_CONTEXT", "32k"),
    "kv": "int8",                 # 8 bits: buena calidad y poco VRAM
    "vision": "off",              # libera ~0.9 GB de VRAM para expertos
    "mtp": "on",                  # el original lo tenia en off
    "drafting": "mtp",            # mtp | dflash2 (experimento) | auto | off
    "spec": _env_int("SK_SPEC", 4),          # propuestas por paso (4 es el valor medido)
    "spec_min_p": _env_float("SK_SPEC_MIN_P", 0.5),
    "mtp_q4": "all",              # draft en 4 bits: +3.5% decode, mismas respuestas
    "prefill": "auto",
    "expert_cache": "auto",       # NUNCA un numero fijo: puede tirar el decode hasta 7x
    "vram_reserve_mib": 512,      # Kaggle no tiene escritorio: reserva VRAM al minimo

    # --- memoria del host (Kaggle: ~31 GB; el Coder necesita ~23 GB de expertos) ---
    "resident_mode": os.environ.get("SK_RESIDENT", "budget"),  # budget | all | off
    "resident_reserve_gib": 14,   # RAM que se deja libre para SO, engine y notebook
    "arena_mmap": os.environ.get("SK_ARENA_MMAP", "0") == "1",  # modo documentado para 2x16 GB + 32 GB RAM
    "pipeline_windows": _env_int("SK_PIPELINE_WINDOWS", 0),   # 2 = experimento NVIDIA
    "pool_workers": _env_int("SK_POOL_WORKERS", 0),           # 0 = auto

    # --- Turing/T4: acelera el prompt, pero cambia el redondeo ---
    "bf16_tc": os.environ.get("SK_BF16_TC", "0") == "1",

    # --- arranque ---
    "tune": "skip",               # en Kaggle no hay tiempo para 20 min de tuning
    "port": 8080,
    "access": os.environ.get("SK_ACCESS", "cloudflare"),  # cloudflare | local
    "show_key": os.environ.get("SK_SHOW_KEY", "0") == "1",  # revelar la API key

    # --- guardas ---
    "disk_free_floor_mb": 8000,   # aborta antes de llenar el disco
    "ram_floor_mb": 1500,         # aborta antes de que el OOM mate la sesion
    "ready_timeout_s": 1800,
    "bench_runs": 3,
}

BASE = f"http://127.0.0.1:{CONFIG['port']}"
print("Configuracion lista. Modelo:", CONFIG["model"].upper(), "| MTP:", CONFIG["mtp"],
      "| spec:", CONFIG["spec"], "| resident:", CONFIG["resident_mode"])


## 2. Comprobar la máquina

Sin esta comprobación, el fallo típico es arrancar con 1 GPU o con el disco ya lleno y descubrirlo 40 minutos después.


In [ ]:
import json, shutil, subprocess, time
from pathlib import Path

import requests

HOME = Path.home()
STRATA_BIN = HOME / ".local" / "bin" / "strata"
os.environ["PATH"] = str(HOME / ".local" / "bin") + os.pathsep + os.environ["PATH"]


def sh(cmd, timeout=120):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return p.returncode, (p.stdout or "").strip(), (p.stderr or "").strip()
    except Exception as exc:
        return 1, "", str(exc)


def meminfo():
    info = {}
    with open("/proc/meminfo") as fh:
        for line in fh:
            key, value = line.split(":", 1)
            info[key] = int(value.strip().split()[0]) // 1024  # MiB
    return info


def disk_gib(path):
    usage = shutil.disk_usage(str(path))
    return usage.total // 2**30, usage.free // 2**30


rc, gpu, err = sh(["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version",
                   "--format=csv,noheader"])
ram = meminfo()
total_gib, free_gib = disk_gib(HOME)
ram_total_gib = ram.get("MemTotal", 0) / 1024
ram_free_gib = ram.get("MemAvailable", 0) / 1024

print("GPUs:")
print(gpu or f"  nvidia-smi no responde: {err}")
print(f"\nRAM total: {ram_total_gib:.1f} GiB | disponible: {ram_free_gib:.1f} GiB")
print(f"Disco home: {total_gib} GiB total, {free_gib} GiB libres")
print(f"CPU: {os.cpu_count()} hilos")

ngpu = len([l for l in (gpu or "").splitlines() if l.strip()])
if ngpu != 2:
    print(f"\nAVISO: se esperaban 2 GPU (T4 x2) y hay {ngpu}. Con una sola GPU el modelo cabe peor y sera mas lento.")
else:
    print("\n2 GPU detectadas: layer-split automatico activado.")

if free_gib < 64:
    raise RuntimeError(
        f"Solo quedan {free_gib} GiB libres. El pack IQ1_M (~58 GB) + draft MTP (~6 GB) no cabe. "
        "Reinicia la sesion de Kaggle para tener disco limpio.")
if free_gib < 72:
    print(f"\nAVISO: quedan {free_gib} GiB libres y el pack + draft ronda los 65 GB. "
          "La guarda de disco de la celda 5 abortara si hace falta.")
if ram_free_gib < 8:
    raise RuntimeError(f"Solo quedan {ram_free_gib:.1f} GiB de RAM. Cierra procesos o reinicia la sesion.")

print("\nPreflight OK.")


## 3. Instalar Strata (solo si falta)

Idempotente: si ya está instalado, no descarga nada.


In [ ]:
if STRATA_BIN.exists() or shutil.which("strata"):
    binary = str(STRATA_BIN) if STRATA_BIN.exists() else "strata"
    rc, out, err = sh([binary, "--version"], timeout=60)
    print("Strata ya instalado:", out or err)
else:
    print("Instalando Strata (una vez por sesion)...")
    rc, out, err = sh([
        "bash", "-lc",
        "curl -fsSL --connect-timeout 15 --max-time 120 "
        "https://github.com/Niko1221/Strata/releases/latest/download/install.sh "
        "-o /tmp/install-strata.sh && sh /tmp/install-strata.sh",
    ], timeout=900)
    print(out[-2000:])
    if err:
        print(err[-2000:])
    if rc != 0:
        raise RuntimeError("La instalacion de Strata fallo. Revisa la salida de arriba.")
    print("Strata instalado.")


## 4. Instalar cloudflared (túnel de Cloudflare)

Strata **no** descarga `cloudflared`: lo busca en el PATH y en las rutas habituales. Esta celda lo instala una vez
por sesión, solo si `CONFIG["access"]` es `cloudflare`. Es idempotente.

Si prefieres una sesión privada, pon `SK_ACCESS=local` y esta celda no hace nada.


In [ ]:
# ============================================================
#  cloudflared: lo necesita el tunel; Strata no lo descarga solo
# ============================================================
import platform

CLOUDFLARED = HOME / ".local" / "bin" / "cloudflared"


def cloudflared_version():
    if not CLOUDFLARED.exists():
        return None
    rc, out, err = sh([str(CLOUDFLARED), "--version"], timeout=30)
    text = (out or err).strip()
    return text.splitlines()[0] if rc == 0 and text else None


version = cloudflared_version()
if CONFIG["access"] != "cloudflare":
    print("Acceso local: no hace falta cloudflared.")
elif version:
    print("cloudflared ya instalado:", version)
else:
    machine = platform.machine().lower()
    arch = {"x86_64": "amd64", "amd64": "amd64", "aarch64": "arm64", "arm64": "arm64"}.get(machine)
    if not arch:
        raise RuntimeError(f"Arquitectura no soportada por cloudflared: {machine}")
    url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}"
    print("Descargando cloudflared:", url)
    rc, out, err = sh([
        "bash", "-lc",
        f"curl -fL --connect-timeout 15 --max-time 180 {url} -o {CLOUDFLARED} && chmod 755 {CLOUDFLARED}",
    ], timeout=300)
    if err:
        print(err[-800:])
    version = cloudflared_version()
    if not version:
        raise RuntimeError(
            "cloudflared no quedo instalado y el tunel no puede arrancar. "
            "Revisa la descarga o arranca en local con SK_ACCESS=local.")
    print("cloudflared instalado:", version)


## 5. Plan (dry run) y validación

`--plan` no descarga ni arranca nada. Debe nombrar **Coder IQ1_M**. Si nombra Swift o IQ2_XS, ese pack no cabe
en el disco de Kaggle: este notebook se detiene.


In [ ]:
import shlex


def resident_budget_gib():
    """GiB de expertos que se copian a RAM, dejando RAM para el SO y el notebook."""
    total = meminfo().get("MemTotal", 0) / 1024
    return max(8, min(24, int(total - CONFIG["resident_reserve_gib"])))


def launch_argv(plan=False):
    """El unico sitio que construye el comando: plan y arranque no pueden divergir."""
    c = CONFIG
    argv = [
        "strata", "launch",
        "--family", c["family"], "--model", c["model"],
        "--context", c["context"], "--kv", c["kv"],
        "--vision", c["vision"],
        "--mtp", c["mtp"], "--drafting", c["drafting"],
        "--tune", c["tune"],
        "--access", c["access"], "--port", str(c["port"]),
    ]

    sets = [
        f"spec={c['spec']}",
        f"spec-min-p={c['spec_min_p']}",
        f"mtp-q4={c['mtp_q4']}",
        f"prefill={c['prefill']}",
        f"expert-cache={c['expert_cache']}",
        f"vram-reserve-mib={c['vram_reserve_mib']}",
    ]
    if c["arena_mmap"]:
        # Documentado para "2x 16 GB GPUs with 32 GB of RAM": mapea la arena en vez de
        # copiarla a RAM bloqueada, y --pcie-frac 0 da los tokens exactos del modo mapeado.
        sets += ["STRATA_ARENA_MMAP=1", "pcie-frac=0"]
    elif c["resident_mode"] == "all":
        sets.append("resident-experts=on")      # booleano: sin valor, o =on
    elif c["resident_mode"] == "budget":
        sets.append(f"resident-budget-gib={resident_budget_gib()}")
    if c["pipeline_windows"]:
        sets.append(f"pipeline-windows={c['pipeline_windows']}")
    if c["pool_workers"]:
        sets.append(f"pool-workers={c['pool_workers']}")
    if c["bf16_tc"]:
        sets.append("STRATA_BF16_TC=1")

    for setting in sets:
        argv += ["--set", setting]

    if plan:
        argv.append("--plan")
    else:
        argv += ["--yes", "--no-tui"]
    return argv


argv_plan = launch_argv(plan=True)
print("Comando (dry run):\n  " + shlex.join(argv_plan) + "\n")
rc, out, err = sh(argv_plan, timeout=900)
plan = out + "\n" + err
print(plan[-4000:])

low = plan.lower()
if "swift" in low or "iq2_xs" in low:
    raise RuntimeError("El plan eligio Swift o IQ2_XS: ese pack no cabe en Kaggle. "
                       "No arranques; revisa --model iq1_m --family coder.")
if "iq1_m" not in low:
    raise RuntimeError("El plan no menciona iq1_m. No arranco a ciegas: revisa la salida de arriba.")
mode = ("tunel de Cloudflare (URL publica con API key)"
        if CONFIG["access"] == "cloudflare" else "solo local (127.0.0.1)")
print("Acceso:", mode)
print("\nPlan validado: Coder IQ1_M. MTP y ajustes de velocidad incluidos.")


## 6. Arrancar con guardas

`strata launch --no-tui --yes` descarga, arranca, imprime la dirección y **devuelve** el control; el servidor
sigue en segundo plano. La celda vigila disco y RAM mientras tanto y aborta antes de romper la sesión.

Si ya hay un servidor respondiendo en el puerto, no relanza nada.


In [ ]:
import queue, signal, threading
from datetime import datetime


def http_model(timeout=3):
    try:
        import requests
        response = requests.get(f"{BASE}/v1/models", timeout=timeout)
        if response.ok:
            data = response.json().get("data") or []
            return data[0]["id"] if data else "?"
    except Exception:
        pass
    return None


def _pump(stream, q):
    for line in stream:
        q.put(line)
    q.put(None)


def _stop_process(pid):
    try:
        os.killpg(os.getpgid(pid), signal.SIGKILL)
    except Exception:
        pass


def launch_with_guards():
    already = http_model()
    if already:
        print(f"Ya hay un servidor sirviendo '{already}' en {BASE}: no se relanza nada.")
        return False

    argv = launch_argv()
    print("Lanzando:\n  " + shlex.join(argv) + "\n")
    log = open("strata.log", "w")
    proc = subprocess.Popen(
        argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy(), start_new_session=True,
    )
    q = queue.Queue()
    threading.Thread(target=_pump, args=(proc.stdout, q), daemon=True).start()

    last_check = 0.0
    try:
        while True:
            try:
                line = q.get(timeout=5)
            except queue.Empty:
                line = ""

            if line is None:
                break
            if line:
                log.write(line)
                log.flush()
                text = line.rstrip()
                if text and any(k in text.lower() for k in
                                ("download", "ready", "serving", "engine", "error",
                                 "draft", "mtp", "expert", "gpu")):
                    print(text[:170])

            now = time.time()
            if now - last_check >= 20:
                last_check = now
                free_mb = shutil.disk_usage(str(HOME)).free // 2**20
                avail_mb = meminfo().get("MemAvailable", 0)
                print(f"  [{datetime.now():%H:%M:%S}] vivo | disco libre {free_mb/1024:.1f} GiB "
                      f"| RAM libre {avail_mb/1024:.1f} GiB")
                if free_mb < CONFIG["disk_free_floor_mb"]:
                    _stop_process(proc.pid)
                    raise RuntimeError(
                        f"Disco casi lleno ({free_mb/1024:.1f} GiB libres). Proceso detenido a tiempo. "
                        "Reinicia la sesion para recuperar disco.")
                if avail_mb < CONFIG["ram_floor_mb"]:
                    _stop_process(proc.pid)
                    raise RuntimeError(
                        f"RAM casi agotada ({avail_mb/1024:.1f} GiB). Proceso detenido antes del OOM. "
                        "Baja SK_RESIDENT o pon SK_RESIDENT=off.")
        rc = proc.wait()
    finally:
        log.close()

    print(f"\nstrata launch termino con codigo {rc}.")
    print("\n".join(open("strata.log").read().splitlines()[-12:]))

    model_id = http_model()
    if model_id:
        print(f"\nLISTO: {model_id} sirviendo en {BASE}")
        return True

    # El comando puede volver antes de que el motor acepte peticiones.
    deadline = time.time() + CONFIG["ready_timeout_s"]
    while time.time() < deadline:
        model_id = http_model()
        if model_id:
            print(f"\nLISTO: {model_id} sirviendo en {BASE}")
            return True
        time.sleep(5)

    if rc != 0:
        raise RuntimeError("El arranque fallo. Mira strata.log.")
    raise RuntimeError("El proceso arranco pero /v1/models no responde. Mira strata.log.")


launched = launch_with_guards()


## 7. Verificar que MTP y los ajustes están **realmente** aplicados

Un flag escrito en el comando no garantiza que el motor lo haya aceptado (Strata avisa y sigue). Esta celda lee el
run config que el launcher escribió y comprueba los valores efectivos.


In [ ]:
import glob

rc, out, err = sh(["strata", "launch", "--status"], timeout=60)
print(out or err)

candidates = []
for root in (HOME / ".config/strata", HOME / ".local/state/strata",
             HOME / ".local/share/strata", HOME / ".strata"):
    candidates += glob.glob(str(root / "**" / "launch" / "*.json"), recursive=True)

if not candidates:
    print("\nNo encontre el run config. Usa la salida de 'strata launch --status' de arriba.")
else:
    cfg_path = max(candidates, key=os.path.getmtime)
    cfg = json.loads(Path(cfg_path).read_text())
    args = [str(a) for a in (cfg.get("args") or [])]
    env = {str(k): str(v) for k, v in (cfg.get("env") or {}).items()}

    def value_of(flag):
        return args[args.index(flag) + 1] if flag in args and args.index(flag) + 1 < len(args) else None

    checks = {
        "MTP activo (--mtp)": "--mtp" in args,
        f"spec = {CONFIG['spec']}": value_of("--spec") == str(CONFIG["spec"]),
        "spec-min-p": value_of("--spec-min-p") == str(CONFIG["spec_min_p"]),
        "mtp-q4 = all": value_of("--mtp-q4") == "all",
        "expert-cache = auto": value_of("--expert-cache") == "auto",
        "expertos residentes en RAM": ("--resident-experts" in args
                                       or value_of("--resident-budget-gib") is not None
                                       or env.get("STRATA_ARENA_MMAP") == "1"),
        "vision apagada": str(cfg.get("vision", "")).lower() in ("false", "none", "off", "0", ""),
    }

    print("\nrun config:", cfg_path)
    for name, ok in checks.items():
        print(("  OK     " if ok else "  FALTA  ") + name)
    print("\nargs del motor:\n  " + " ".join(args)[:1400])
    if env:
        print("\nenv del motor:\n  " + " ".join(f"{k}={v}" for k, v in env.items())[:800])

    missing = [name for name, ok in checks.items() if not ok]
    if missing:
        print("\nAVISO: el motor no reporta estos ajustes:", ", ".join(missing))
        print("Strata escribe un aviso cuando rechaza un ajuste; buscalo en strata.log.")
    else:
        print("\nTodos los ajustes de velocidad estan activos en el motor.")


## 8. Conexión: URL pública y API key

Con `access = "cloudflare"` el servidor escucha solo en `127.0.0.1` y sale al exterior por un *quick tunnel* de
Cloudflare. Strata crea una **API key** automáticamente; aquí se muestra enmascarada.

La celda reintenta hasta 90 s: Strata espera hasta 45 s a que `cloudflared` publique la URL.


In [ ]:
# ============================================================
#  Conexion: URL publica del tunel y API key
# ============================================================
import glob
import re

TUNNEL_URL = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
KEY_RE = re.compile(r"sk-[A-Za-z0-9_-]{8,}")

PUBLIC_BASE = None
API_KEY = None
MODEL_ID = http_model()


def find_tunnel_url():
    rc, out, err = sh(["strata", "launch", "--status"], timeout=60)
    status = (out + chr(10) + err).strip()
    match = TUNNEL_URL.search(status)
    if match:
        return match.group(0), status
    for root in (HOME / ".config/strata", HOME / ".local/state/strata",
                 HOME / ".local/share/strata", HOME / ".strata"):
        for path in glob.glob(str(root / "**" / "*.tunnel.json"), recursive=True):
            try:
                data = json.loads(Path(path).read_text())
            except Exception:
                continue
            if data.get("url"):
                return str(data["url"]), status
    return None, status


if CONFIG["access"] == "cloudflare":
    url, status = None, ""
    deadline = time.time() + 90
    while time.time() < deadline and not url:
        url, status = find_tunnel_url()
        if not url:
            time.sleep(5)
    if url:
        PUBLIC_BASE = url.rstrip("/") + "/v1"
        print("URL publica :", PUBLIC_BASE)
    else:
        print("No se encontro la URL del tunel.")
        if status:
            print(status[-1200:])
        print("Comprueba: ~/.local/bin/cloudflared --version  y  strata.log")
else:
    PUBLIC_BASE = BASE + "/v1"
    print("Acceso local:", PUBLIC_BASE)

if not MODEL_ID:
    MODEL_ID = http_model()
print("Modelo      :", MODEL_ID or "(sin respuesta)")

rc, out, err = sh(["strata", "launch", "--show-key"], timeout=60)
match = KEY_RE.search(out or "")
API_KEY = match.group(0) if match else ((out or "").strip().splitlines()[-1] if (out or "").strip() else None)

if API_KEY:
    if CONFIG["show_key"]:
        print("API key     :", API_KEY)
    else:
        print("API key     :", API_KEY[:6] + "********" + API_KEY[-2:], " (revelar: SK_SHOW_KEY=1)")
else:
    print("API key     : (sin key)")


## 9. Conectar una app

Cualquier cliente compatible con OpenAI sirve. Cambia `<API_KEY>` por la key real (`strata launch --show-key`).

El benchmark y el SVG de las secciones siguientes siguen usando `127.0.0.1` a propósito: el túnel añade latencia
y no cambia la velocidad del motor.


In [ ]:
# ============================================================
#  Cliente: OpenAI SDK / curl
# ============================================================
base = PUBLIC_BASE or (BASE + "/v1")
if API_KEY and not CONFIG["show_key"]:
    shown = API_KEY[:6] + "********" + API_KEY[-2:]
else:
    shown = API_KEY or "(sin key)"

print("base_url :", base)
print("model    :", MODEL_ID)
print("api_key  :", shown)
print("")
print("# OpenAI SDK")
print(f'client = OpenAI(base_url="{base}", api_key="<API_KEY>")')
print(f'client.chat.completions.create(model="{MODEL_ID}", messages=[{{"role": "user", "content": "hola"}}])')
print("")
print("# curl (sustituye <API_KEY> por la key real)")
example = '{"model": "%s", "messages": [{"role": "user", "content": "hola"}]}' % MODEL_ID
print(f'curl {base}/chat/completions -H "Authorization: Bearer <API_KEY>" '
      f'-H "Content-Type: application/json" -d ' + shlex.quote(example))
print("")
print("Key real         : strata launch --show-key")
print("Cerrar servidor  : strata launch --stop (tambien cierra el tunel)")
print("Aviso: la URL es publica; la API key es la unica barrera.")


## 10. Benchmark: tokens por segundo reales

Calentamiento + 3 pasadas con el mismo prompt y `temperature=0`. Se reporta la **mediana** del decode, que es la
cifra honesta: la primera pasada siempre paga la carga inicial.


In [ ]:
import statistics

PROMPT = ("Write a Python function merge_intervals(intervals) that merges overlapping "
          "closed intervals and returns them sorted. Include a short docstring.")


def one_run(max_tokens=256, keep_text=False):
    import requests
    model_id = http_model()
    if not model_id:
        raise RuntimeError("No hay modelo respondiendo en " + BASE)

    payload = {
        "model": model_id,
        "messages": [{"role": "user", "content": PROMPT}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": True,
        "stream_options": {"include_usage": True},
    }

    t0 = time.time()
    first = None
    chunks = 0
    usage = None
    content = []

    with requests.post(f"{BASE}/v1/chat/completions", json=payload, stream=True, timeout=3600) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:"):
                continue
            chunk = line[5:].strip()
            if chunk == "[DONE]":
                break
            try:
                obj = json.loads(chunk)
            except json.JSONDecodeError:
                continue
            if obj.get("usage"):
                usage = obj["usage"].get("completion_tokens")
            choices = obj.get("choices") or []
            if not choices:
                continue
            delta = choices[0].get("delta") or {}
            piece = delta.get("content") or delta.get("reasoning_content") or ""
            if piece:
                if first is None:
                    first = time.time()
                chunks += 1
                if keep_text:
                    content.append(piece)

    end = time.time()
    if first is None:
        return None
    tokens = usage or chunks
    result = {"ttft": first - t0, "wall": end - t0, "tokens": tokens,
              "tps": tokens / max(end - first, 1e-6)}
    if keep_text:
        result["text"] = "".join(content)
    return result


print("Calentamiento...")
warm = one_run(max_tokens=64, keep_text=False)
if warm:
    print(f"  {warm['tps']:.2f} tok/s (no cuenta)")

rows = []
for i in range(CONFIG["bench_runs"]):
    row = one_run(keep_text=(i == 0))
    if not row:
        continue
    rows.append(row)
    print(f"  run {i+1}: {row['tps']:6.2f} tok/s | TTFT {row['ttft']:5.2f}s "
          f"| {row['tokens']} tokens | {row['wall']:.1f}s")

if rows:
    median_tps = statistics.median(r["tps"] for r in rows)
    median_ttft = statistics.median(r["ttft"] for r in rows)
    print(f"\nDecode (mediana de {len(rows)}): {median_tps:.2f} tok/s")
    print(f"TTFT   (mediana): {median_ttft:.2f} s")
    print(f"\nCon MTP el motor verifica hasta {CONFIG['spec']} propuestas por paso de pesos.")
    print("Si este numero te decepciona, mira la RAM libre: con expertos en SSD el decode cae en picado.")
    if rows[0].get("text"):
        print("\nPrimeras lineas de la respuesta:\n" + rows[0]["text"][:600])


## 11. Opcional: dibujar un SVG con vista previa en vivo

Cambia `prompt` por lo que quieras. El archivo se guarda en `/kaggle/working/drawing.svg`.


In [ ]:
import re
from IPython.display import HTML, display, update_display

prompt = ("A cute astronaut floating in space with a dark purple background, small stars, "
          "a ringed planet, and a crescent moon. Flat colors, rounded shapes.")

model_id = http_model()
if not model_id:
    raise RuntimeError("No hay modelo respondiendo en " + BASE)

payload = {
    "model": model_id,
    "messages": [
        {"role": "system", "content": "Reply with one complete SVG only (viewBox 0 0 400 400), no explanation."},
        {"role": "user", "content": prompt},
    ],
    "max_tokens": 6000,
    "temperature": 0.6,
    "stream": True,
    "stream_options": {"include_usage": True},
}


def extract_svg(text):
    match = re.search(r"<svg\b.*?</svg>", text, re.S)
    if match:
        return match.group(0)
    start = text.find("<svg")
    return text[start:] if start != -1 else ""


def panel(state, tokens, tps, elapsed, svg):
    return HTML(
        '<div style="font-family:monospace;font-size:13px;margin-bottom:8px">'
        f'<b>Model:</b> {payload["model"]} &nbsp;|&nbsp; <b>State:</b> {state} &nbsp;|&nbsp; '
        f'<b>Tokens:</b> {tokens} &nbsp;|&nbsp; <b>Speed:</b> {tps:.2f} tok/s &nbsp;|&nbsp; '
        f'<b>Time:</b> {elapsed:.0f}s</div>'
        f'<div style="width:400px;max-width:100%;background:#fff;border:1px solid #888">{svg}</div>'
    )


handle = display(panel("waiting", 0, 0, 0, ""), display_id=True)
t0 = time.time()
first = None
content = []
tokens = 0
usage = None
last_draw = 0.0

with requests.post(f"{BASE}/v1/chat/completions", json=payload, stream=True, timeout=3600) as response:
    response.raise_for_status()
    for line in response.iter_lines(decode_unicode=True):
        if not line or not line.startswith("data:"):
            continue
        chunk = line[5:].strip()
        if chunk == "[DONE]":
            break
        try:
            obj = json.loads(chunk)
        except json.JSONDecodeError:
            continue
        if obj.get("usage"):
            usage = obj["usage"].get("completion_tokens")
        choices = obj.get("choices") or []
        if not choices:
            continue
        delta = choices[0].get("delta") or {}
        think = delta.get("reasoning_content") or ""
        piece = delta.get("content") or ""
        if not (think or piece):
            continue
        if first is None:
            first = time.time()
        tokens += 1
        content.append(piece)
        now = time.time()
        if now - last_draw > 0.4:
            text = "".join(content)
            state = "drawing" if piece else "thinking"
            update_display(panel(state, tokens, tokens / max(now - first, 0.01), now - t0, extract_svg(text)),
                           display_id=handle.display_id)
            last_draw = now

end = time.time()
final_tokens = usage or tokens
tps = final_tokens / max(end - (first or t0), 0.01)
svg = extract_svg("".join(content))
update_display(panel("done", final_tokens, tps, end - t0, svg), display_id=handle.display_id)

if svg.endswith("</svg>"):
    open("/kaggle/working/drawing.svg", "w").write(svg)
    print(f"Guardado en /kaggle/working/drawing.svg | {final_tokens} tokens | {tps:.2f} tok/s")
else:
    print("El SVG parece cortado: sube max_tokens y repite.")


## 12. Parar el servidor y el túnel

Ejecútalo al terminar para liberar VRAM y cerrar la URL pública, o simplemente cierra la sesión de Kaggle.


In [ ]:
rc, out, err = sh(["strata", "launch", "--stop"], timeout=120)
print(out or err or "Servidor y tunel detenidos.")
